In [ ]:
# Install dependencies
!pip install xarray zarr fsspec ipfsspec matplotlib

In [ ]:
# Import Libraries
import xarray as xr
import zarr
import fsspec
import ipfsspec
import matplotlib.pyplot as plt
import os

In [ ]:
os.environ["IPFS_GATEWAY"] = "https://ipfs.io"

# Accessing the data via the IPFS
data_url = "ipfs://bafybeihfqxfckruepjhrkafaz6xg5a4sepx6ahhv4zds4b3hnfiyj35c5i"

print(f"Attempting to access data from: {data_url}")

try:
    ds = xr.open_dataset(data_url, engine="zarr")
    print("Dataset successfully loaded!\n")
    
    # Display an overview of the data structure
    display(ds)
except Exception as e:
    print(f"Error: {e}")

In [ ]:
start_time = ds.circle_time.min().values
end_time = ds.circle_time.max().values

print(f"Data Start Date : {start_time}")
print(f"Data End Date   : {end_time}")

In [ ]:
# Mean vertical velocity profile
var_w = 'wvel'
var_omega = 'omega'
omega_mean = ds[var_omega].mean(dim='circle', keep_attrs=True)

# Plot the vertical profile
plt.figure(figsize=(6, 8))

# Plotting against the 'altitude' dimension on the y-axis
omega_mean.plot(y='altitude', color='b', linewidth=2)

# Formatting the plot
plt.title("Mean Vertical Ascent Profile (omega)")
plt.xlabel("Vertical Velocity (Pa/s)")
plt.ylabel("Altitude (m)")
plt.axvline(x=0, color='k', linestyle='--', alpha=0.7) # Zero-reference line
plt.grid(True, linestyle=':')
plt.tight_layout()
plt.show()

In [ ]:
# 1. Choose a specific date
selected_date = "2024-08-13"

# 2. Filter the dataset for that date
ds_subset = ds.sel(circle_time=slice(f"{selected_date}T00:00", f"{selected_date}T23:59"))

if len(ds_subset.circle) > 0:
    print(f"Found {len(ds_subset.circle)} circles for {selected_date}")
    
    # 3. Calculate mean omega and mean pressure for the Y-axis
    daily_omega = ds_subset['omega'].mean(dim='circle')
    # Use 'p_mean' as the vertical coordinate (Pressure in Pa)
    p_coord = ds_subset['p_mean'].mean(dim='circle') 
    
    plt.figure(figsize=(7, 9))
    
    # 4. Plot omega against Pressure
    # Note: We convert p_coord to hPa (hectopascals) by dividing by 100 for better readability
    plt.plot(daily_omega, p_coord/100, color='red', linewidth=2.5, label=f'Mean {selected_date}')
    
    # 5. Add Campaign Average for context
    campaign_omega = ds['omega'].mean(dim='circle')
    campaign_p = ds['p_mean'].mean(dim='circle')
    plt.plot(campaign_omega, campaign_p/100, color='black', linestyle='--', alpha=0.3, label='Campaign Average')

    # 6. Formatting: INVERT Y-AXIS (Crucial for Pressure plots)
    plt.gca().invert_yaxis() 
    
    plt.title(f"Vertical Motion Profile: {selected_date}", fontsize=14)
    plt.axvline(x=0, color='black', linewidth=1)
    plt.xlabel("Omega [Pa/s]")
    plt.ylabel("Pressure [hPa]")
    plt.legend()
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.show()
else:
    print(f"No data available for {selected_date}.")

In [ ]:
import pandas as pd

# 1. Define the full range of dates based on our previous metadata check
start_dt = pd.to_datetime(ds.circle_time.min().values).date()
end_dt = pd.to_datetime(ds.circle_time.max().values).date()

# Create a list of all calendar days in between
all_days = pd.date_range(start=start_dt, end=end_dt, freq='D')

available_dates = []
missing_dates = []

print(f"Auditing data from {start_dt} to {end_dt}...\n")

# 2. Loop through each day and check the dataset
for day in all_days:
    date_str = day.strftime('%Y-%m-%d')
    # Slice the data for the current 24-hour window
    day_data = ds.sel(circle_time=slice(f"{date_str}T00:00", f"{date_str}T23:59"))
    
    if len(day_data.circle) > 0:
        available_dates.append(date_str)
    else:
        missing_dates.append(date_str)

# 3. Print the Summary
print(f"--- Audit Summary ---")
print(f"Total Campaign Days   : {len(all_days)}")
print(f"Days WITH Data        : {len(available_dates)}")
print(f"Days WITHOUT Data     : {len(missing_dates)}")
print(f"----------------------")


In [ ]:
# 1. Prepare the figure
plt.figure(figsize=(10, 12))

# 2. Loop through each available date and plot
print(f"Plotting profiles for {len(available_dates)} days...")

for date_str in available_dates:
    # Filter data for the specific day
    ds_day = ds.sel(circle_time=slice(f"{date_str}T00:00", f"{date_str}T23:59"))
    
    # Calculate mean omega and pressure for the day
    daily_omega = ds_day['omega'].mean(dim='circle')
    daily_p = ds_day['p_mean'].mean(dim='circle')
    
    # Plot each day with a thin line and low alpha (transparency)
    # This creates a "spaghetti plot" effect to show variability
    plt.plot(daily_omega, daily_p/100, alpha=0.3, linewidth=1, label='_nolegend_')

# 3. Add the Campaign Average (The "Reference" Profile)
campaign_omega = ds['omega'].mean(dim='circle')
campaign_p = ds['p_mean'].mean(dim='circle')

plt.plot(campaign_omega, campaign_p/100, color='black', linewidth=4, 
         label=f'Campaign Average ({len(available_dates)} days)')

# 4. Final Formatting
plt.gca().invert_yaxis() # Put 1000 hPa at the bottom
plt.axvline(x=0, color='red', linestyle='-', linewidth=1.5, alpha=0.7) # Zero line

plt.title("Vertical Motion Profiles (Omega) - All ORCESTRA Missions", fontsize=15)
plt.xlabel("Pressure Velocity (omega) [Pa/s] \n <--- Ascent | Descent --->", fontsize=12)
plt.ylabel("Pressure Level [hPa]", fontsize=12)
plt.legend(loc='upper left')
plt.grid(True, linestyle=':', alpha=0.6)

# Set X-axis limits to focus on the data (adjust if needed)
plt.xlim(-0.8, 0.4) 

plt.tight_layout()
plt.show()

In [ ]:
import math

# 1. Calculate grid dimensions based on available dates
num_plots = len(available_dates)
cols = 4  # You can adjust the number of columns (e.g., 3 or 5)
rows = math.ceil(num_plots / cols)

# 2. Create the figure and subplots
fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 5), sharey=True)
axes = axes.flatten() # Flatten to loop easily

print(f"Generating {num_plots} daily time-series plots...")

# 3. Loop through each date and plot in its own subplot
for i, date_str in enumerate(available_dates):
    ax = axes[i]
    
    # Filter data for the specific day
    ds_day = ds.sel(circle_time=slice(f"{date_str}T00:00", f"{date_str}T23:59"))
    
    # Calculate daily mean
    daily_omega = ds_day['omega'].mean(dim='circle')
    daily_p = ds_day['p_mean'].mean(dim='circle') / 100 # Convert to hPa
    
    # Plotting
    ax.plot(daily_omega, daily_p, color='blue', linewidth=2)
    ax.axvline(x=0, color='black', linestyle='-', linewidth=0.8, alpha=0.5)
    
    # Formatting each subplot
    ax.set_title(f"Date: {date_str}", fontsize=10, fontweight='bold')
    ax.invert_yaxis() # Highest pressure at bottom
    ax.grid(True, linestyle=':', alpha=0.4)
    
    # Set axis limits for comparison
    ax.set_xlim(-1.0, 0.5) 
    if i % cols == 0:
        ax.set_ylabel("Pressure [hPa]")
    if i >= num_plots - cols:
        ax.set_xlabel("Omega [Pa/s]")

# 4. Remove empty subplots (if any)
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.suptitle("Daily Evolution of Vertical Motion - ORCESTRA Campaign", fontsize=20, y=1.02)
plt.show()

In [ ]:
import math

# 1. Preparation: Calculate the global mean across all circles first
campaign_omega = ds['omega'].mean(dim='circle')
campaign_p = ds['p_mean'].mean(dim='circle') / 100

# 2. Grid Setup
# We add +1 to num_plots to make room for the "Mean" chart
num_plots = len(available_dates) + 1 
cols = 4
rows = math.ceil(num_plots / cols)

fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 5), sharey=True)
axes = axes.flatten()

print(f"Generating grid with {len(available_dates)} daily plots + 1 Campaign Mean...")

# 3. Plot the Campaign Mean in the FIRST slot (Index 0)
ax_mean = axes[0]
ax_mean.plot(campaign_omega, campaign_p, color='black', linewidth=3, label='Campaign Mean')
ax_mean.axvline(x=0, color='red', linestyle='-', linewidth=1, alpha=0.5)
ax_mean.set_title("OVERALL CAMPAIGN MEAN", fontsize=10, fontweight='bold', color='red')
ax_mean.invert_yaxis()
ax_mean.set_ylabel("Pressure [hPa]")
ax_mean.grid(True, linestyle='--', alpha=0.6)
ax_mean.set_xlim(-0.8, 0.4) # Consistent scale

# 4. Loop through available dates for the remaining slots
for i, date_str in enumerate(available_dates):
    # Offset by 1 because index 0 is the Mean chart
    ax = axes[i + 1] 
    
    # Filter and calculate daily mean
    ds_day = ds.sel(circle_time=slice(f"{date_str}T00:00", f"{date_str}T23:59"))
    daily_omega = ds_day['omega'].mean(dim='circle')
    daily_p = ds_day['p_mean'].mean(dim='circle') / 100
    
    # Plot daily data
    ax.plot(daily_omega, daily_p, color='blue', linewidth=2)
    ax.axvline(x=0, color='black', linestyle='-', linewidth=0.8, alpha=0.5)
    
    # Formatting
    ax.set_title(f"Date: {date_str}", fontsize=10)
    ax.set_xlim(-0.8, 0.4) 
    ax.grid(True, linestyle=':', alpha=0.4)
    
    # Axis labels for edge subplots
    if (i + 1) % cols == 0:
        ax.set_ylabel("Pressure [hPa]")
    if (i + 1) >= num_plots - cols:
        ax.set_xlabel("Omega [Pa/s]")

# 5. Clean up
for j in range(num_plots, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.suptitle("Vertical Profile (Omega) ORCESTRA Campaign", fontsize=25, y=1.015)
plt.show()